In [11]:
import numpy as np

fs = 50                 # sampling rate (samples per second)
total_time = 1200            # total paradigm duration in seconds


with open(r"J:\project_trainingAggression\Data\20250817_mouse975826\Day19\neuralData\catgt_20250818_m975826_obs6_g0\20250818_m975826_obs6_g0_tcat.nidq.xd_3_4_1e+06.txt", "r") as gate:
    gateF = gate.read()

with open(r"J:\project_trainingAggression\Data\20250817_mouse975826\Day19\neuralData\catgt_20250818_m975826_obs6_g0\20250818_m975826_obs6_g0_tcat.nidq.xd_3_3_0.txt", "r") as redLightOn:
    redL_ON = redLightOn.read()

with open(r"J:\project_trainingAggression\Data\20250817_mouse975826\Day19\neuralData\catgt_20250818_m975826_obs6_g0\20250818_m975826_obs6_g0_tcat.nidq.xid_3_3_360000.txt", "r") as redLightOFF:
    redL_OFF = redLightOFF.read()

# convert gateF to float seconds
gate_open_time = float(gateF.strip())
redL_ON_time = [float(x) for x in redL_ON.split()]
redL_OFF_time = float(redL_OFF.strip())

# total number of samples in the paradigm
n_samples = int(round(total_time * fs))

# sample where gate opens
gate_open_sample = int(round(gate_open_time * fs))
redL_ON_sample = [int(round(f * fs)) for f in redL_ON_time]
redL_OFF_sample = int(round(redL_OFF_time * fs))

# create behavioral matrix / vector
gateOpen = np.zeros(n_samples, dtype=int)
lightOn = np.zeros(n_samples, dtype=int)
lightOff = np.zeros(n_samples, dtype=int)

gateOpen[gate_open_sample:] = 1
lightOn[redL_OFF_sample:redL_ON_sample[1]] = 1

print(gateOpen)
print(gateOpen.shape)

[0 0 0 ... 1 1 1]
(60000,)


In [12]:
behavioral_matrix = {}

behavioral_matrix["gateOpen"] = gateOpen
behavioral_matrix["whiteLight"] = lightOn


In [13]:
def extract_on_periods(signal, fs):
    signal = np.asarray(signal).astype(int)

    # pad with 0 at both ends so transitions are easy to detect
    padded = np.pad(signal, (1, 1), mode="constant", constant_values=0)

    # find changes
    diff = np.diff(padded)

    # starts where 0 -> 1
    starts = np.where(diff == 1)[0]

    # ends where 1 -> 0
    ends = np.where(diff == -1)[0]

    # convert samples to seconds
    periods_sec = [(start / fs, end / fs) for start, end in zip(starts, ends)]

    return starts, ends, periods_sec

for name, signal in behavioral_matrix.items():
    starts, ends, periods_sec = extract_on_periods(signal, fs)

    print(f"\n{name}")
    print("Start samples:", starts)
    print("End samples:  ", ends)
    print("Periods in seconds:")
    for start_s, end_s in periods_sec:
        print(f"  {start_s/60:.3f} -> {end_s/60:.3f}")


gateOpen
Start samples: [17268]
End samples:   [60000]
Periods in seconds:
  5.756 -> 20.000

whiteLight
Start samples: [14265]
End samples:   [32283]
Periods in seconds:
  4.755 -> 10.761


In [14]:
np.save(r"C:\Users\Data Analysis\Desktop\paradigmConditions.npy", behavioral_matrix)